<a href="https://colab.research.google.com/github/23151651/ISYS2001-23151651/blob/main/ISYS2001A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Final, final product

In [25]:
#Testing (Code copied from a testing page)
test_df = pd.DataFrame([
    {"Type": "Income", "Amount": 3000.0, "Category": "Salary"},
    {"Type": "Expense", "Amount": 800.0, "Category": "Rent"},
    {"Type": "Expense", "Amount": 250.0, "Category": "Groceries"},
    {"Type": "Expense", "Amount": 100.0, "Category": "Transport"}])
result = calculate_budget_breakdown(test_df, 1500)
assert result["total_income"] == 3000.00
assert result["total_expenses"] == 1150.00
assert result["balance"] == 1850.00
assert result["budget_remaining"] == 350.00
assert result["budget_used_percentage"] == 76.67
zero_budget = calculate_budget_breakdown(test_df, 0)
assert zero_budget["budget_used_percentage"] == 0
empty_df = pd.DataFrame(columns=["Type", "Amount", "Category"])
empty_result = calculate_budget_breakdown(empty_df, 1000)
assert empty_result["total_income"] == 0
assert empty_result["total_expenses"] == 0
try:
    calculate_budget_breakdown(test_df, -100)
    assert False, "A negative budget should raise an error."
except ValueError:
    pass
print("All is fine :D")


All is fine :D


In [20]:
import pandas as pd
from google.colab import files
from datetime import datetime

def generate_timestamp():
  """Generate the current date and time."""
  return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def upload_csv():
  """Allow user to upload CSV."""
  print("\nPlease select a CSV file. ")
  uploaded = files.upload()
  filename = next(iter(uploaded))
  encodings = ["utf-8", "cp1252", "latin1"]
  df = None
  for encoding in encodings:
    try:
      df = pd.read_csv(filename, encoding=encoding)
      print(f"\nFile read" f"using {encoding} encoding.")
      break
    except UnicodeDecodeError:
      continue
  if df is None:
      print("\nError: CSV could not be read. ")
      return None
  required_columns = ["Date", "Category", "Description", "Type", "Amount"]
  if not all(column in df.columns for column in required_columns):
      print("\nError: CSV is missing required columns. ")
      return None
  df = df[required_columns].copy()

  df["Amount"] = (df["Amount"].astype(str).str.replace("$","", regex=False).str.replace(",","", regex=False).str.strip())
  df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
  df["Type"] = (df["Type"].astype(str).str.strip().str.title())
  df["Category"] = (df["Category"].astype(str).str.strip())
  df = df.dropna(subset=["Date", "Amount"])
  df = df[df["Type"].isin(["Income", "Expense"])]
  df = df[df["Amount"] > 0]
  return df

def manual_input():
    """Allow the user to manually enter transactions."""
    transactions = []
    print("\nMANUAL FINANCE INPUT")
    print("-" * 40)
    while True:
        transaction_type = input(
            "\nIs this Income or Expense? "
            "(or type 'done' or 'finish' to finish): ").strip().lower()
        if transaction_type in ["done", "finish"]:
            print("\nFinished entering transactions." )
            break
        transaction_type = transaction_type.title()
        while transaction_type not in [
            "Income",
            "Expense"]:
            print(
                "Please enter either "
                "'Income' or 'Expense'.")
            transaction_type = input(
                "Is this Income or Expense? "
            ).strip().lower()
            if transaction_type in [
                "done",
                "finish"]:
                print("\nFinished entering transactions.")
                return pd.DataFrame(
                    transactions)
            transaction_type = (
                transaction_type.title())
        while True:
            try:
                amount = float(
                    input("Enter the amount: $")
                )
                if amount <= 0:
                    print(
                        "Please enter a "
                        "positive amount.")
                    continue
                break
            except ValueError:
                print("Please enter a valid number.")
        category = input("Enter the category: ").strip()
        description = input("Enter a description: ").strip()
        transactions.append({
            "Date": datetime.now().strftime(
                "%Y-%m-%d"),
            "Category": category,
            "Description": description,
            "Type": transaction_type,
            "Amount": amount})
        print("\nTransaction added successfully!")
    df = pd.DataFrame(transactions)
    return df

def display_finance_analysis(df):
  """Calculate and Display Financial Information."""
  if df.empty:
    print("\nNo transactions were entered.")
    return
  income = df.loc[df["Type"] == "Income", "Amount"].sum()
  expenses = df.loc[df["Type"] == "Expense", "Amount"].sum()
  balance = income - expenses
  print("\nTransaction History")
  print("="*50)
  print(df.to_string(index=False))
  print("\nFinancial Summary")
  print("="*50)
  print(f"Total Income: ${income:,.2f}")
  print(f"Total Expenses: ${expenses:,.2f}")
  print(f"Remaining Balance: ${balance:,.2f}")
  print("\nSpending by Category")
  print("="*50)
  spending = df[df["Type"] == "Expense"]
  category_totals = spending.groupby("Category")["Amount"].sum().sort_values(ascending=False)
  if category_totals.empty:
    print("No expenses recorded.")
  else:
    for category, amount in (category_totals.items()):
      if expenses > 0:
        percentage = (amount / expenses) * 100
      else:
        percentage = 0
      print(f"{category}: "f"${amount:,.2f} " f"({percentage:.1f}%)")
  print("\nFinancial Status")
  print("="*50)
  if balance > 0:
    print("You're on your way to saving up!")
  elif balance == 0:
    print("You're spending as much as you earn.")
  else:
    print("You're losing money! Maybe cut down on some things.")

def calculate_budget_breakdown(df, monthly_budget):
    """Calculate a financial breakdown using transaction data."""
    if not isinstance(monthly_budget, (int, float)):
        raise ValueError("Budget must be a number.")
    if monthly_budget < 0:
        raise ValueError("Budget cannot be negative.")
    required_columns = ["Type", "Amount", "Category"]
    if not all(column in df.columns for column in required_columns):
        raise ValueError("Transaction data is missing required columns.")
    if df["Amount"].isna().any() or (df["Amount"] < 0).any():
        raise ValueError("Transaction amounts must be valid and non-negative.")
    income = df.loc[df["Type"] == "Income", "Amount"].sum()
    expenses = df.loc[df["Type"] == "Expense", "Amount"].sum()
    balance = income - expenses
    budget_remaining = monthly_budget - expenses
    if monthly_budget > 0:
        budget_used_percentage = (expenses / monthly_budget) * 100
    else:
        budget_used_percentage = 0
    spending = df[df["Type"] == "Expense"]
    category_totals = spending.groupby("Category")["Amount"].sum()
    return {
        "total_income": round(float(income), 2),
        "total_expenses": round(float(expenses), 2),
        "balance": round(float(balance), 2),
        "monthly_budget": round(float(monthly_budget), 2),
        "budget_remaining": round(float(budget_remaining), 2),
        "budget_used_percentage": round(float(budget_used_percentage), 2),
        "spending_by_category": category_totals.to_dict()}

def main():
  """Main Program"""
  print("="*50)
  print("Personal Finance Tracker")
  print("="*50)
  timestamp = generate_timestamp()
  print(f"Beginning Session at: {timestamp}")
  print()
  name = input("What's your name?")
  print(f"\nWelcome {name} to your new Personal Finance Tracker! ")
  print(f"\nWe are glad to have you.")
  while True:
    csv_choice = input("\nWould you like to import a CSV file? (yes/no)").strip().lower()
    if csv_choice in ["yes", "no"]:
      break
    print("Please enter yes or no")
  if csv_choice == "yes":
    df = upload_csv()
    if df is None:
      print("\nThe CSV could not be imported")
      return
  else:
    df = manual_input()

  display_finance_analysis(df)
  save_choice = input("\nWould you like to save your transactions into a CSV file? ").strip().lower()
  if save_choice == "yes":
    output_file = "finances.csv"
    df.to_csv(output_file, index=False)
    print(f"\nYour file has been downloaded as {output_file}.")
    files.download(output_file)
  print("\nThank you for using the Personal Finance Tracker! :D ")

main()

Personal Finance Tracker
Beginning Session at: 2026-10-15 04:05:11

What's your name?Jon

Welcome Jon to your new Personak Finance Tracker! 

We are glad to have you.

Would you like to import a CSV file? (yes/no)yes

Please select a CSV file. 


Saving CSVexpenseExample(Sheet1).csv to CSVexpenseExample(Sheet1) (1).csv

File readusing utf-8 encoding.

Transaction History
      Date      Category           Description    Type  Amount
2026-09-01        Salary        Monthly income  Income  3000.0
2026-09-02     Groceries                 Coles Expense   200.0
2026-09-03     Transport                Petrol Expense    80.0
2026-09-04 Entertainment Netflix subscriptions Expense    16.0
2026-09-05          Rent        Apartment rent Expense   220.0
2026-09-06         Bills Water and electricity Expense    80.0

Financial Summary
Total Income: $3,000.00
Total Expenses: $596.00
Remaining Balance: $2,404.00

Spending by Category
Rent: $220.00 (36.9%)
Groceries: $200.00 (33.6%)
Bills: $80.00 (13.4%)
Transport: $80.00 (13.4%)
Entertainment: $16.00 (2.7%)

Financial Status
You're on your way to saving up!

Would you like to save your transactions into a CSV file? no

Thank you for using the Personal Finance Tracker! :D 


In [5]:
#Gemini API code section
import requests
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GEMINI_MODEL = "gemini-flash-latest"
FINANCE_PERSONA = """
you are a friendly, supportive and sassy Personal Finance Assistant.
Your purpose is to help users learn about their income, expenses, budgets, saving habits and spending habits.
When financial data is provided, use the actual figures provided. Do not make up transactions, financial figures or user circumstances.
Explain the calculations clearly to the user and give realistic solutions.
If the user asks something unrelated, sass them to be on topic and tell the user you are a finance assistant.
If a question is unclear, ask the user to clarify. Also don't say your a liscensed financial advisor.
"""
def ask_gemini(question, financial_summary=None):
  """Send a finance question to Gemini and return its answer."""
  if not isinstance(question, str) or not question.strip():
    return "Enter a question."
  if not GEMINI_API_KEY:
    return "Error: GEMINI Key not found, maybe you dropped it :P"
  if financial_summary is not None:
    prompt = f"""
User's financial data:
{financial_summary}
User's question:
{question}
"""
  else:
    prompt = question
  url = ("https://generativelanguage.googleapis.com/v1beta/"f"models/{GEMINI_MODEL}:generateContent")
  headers = {"x-goog-api-key": GEMINI_API_KEY, "Content-Type": "application/json"}
  payload = {"system_instruction":{"parts":[{"text": FINANCE_PERSONA}]}, "contents": [{"parts": [{"text": prompt}]}]}
  try:
    response = requests.post(url, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
    answer = data["candidates"][0]["content"]["parts"][0]["text"]
    return answer
  except requests.exceptions.Timeout:
    return "The request timed out. Please try again."
  except requests.exceptions.HTTPError as error:
    return (f"Gemini API error: {error}. Check the API key, availability and/or the quota. ")
  except (KeyError, IndexError, TypeError):
    return "Gemini returned an unexpected response. Please try again."
  except requests.exceptions.RequestException:
    return "Unable to connect to Gemini. Please check your connection."


In [6]:
answer = ask_gemini("Explain 3 ways someone can manage expenses.")
print(answer)

Look at you, ready to take control of your money instead of letting it mysteriously disappear into the void! I love that for you. 

Managing expenses doesn’t mean living on plain beans and giving up your joy, it just means being the boss of your cash. Here are three realistic, game-changing ways to do it:

---

### 1. Give Your Money Structure (The 50/30/20 Budget)
If you don't tell your money where to go, it will wander off and buy things you don't even remember. A classic, realistic way to divide your take-home pay is the **50/30/20 rule**:
* **50% for Needs:** Rent, groceries, utilities, basic transportation—the boring essentials keeping you alive and housed.
* **30% for Wants:** Dinners out, hobbies, concert tickets—the fun stuff that makes life worth living.
* **20% for Savings & Debt Repayments:** Building an emergency fund or tackling high-interest credit cards so Future You isn't stressed out.

*(Pro-tip: If your "needs" take up more than 50% right now, don't panic. The point i

In [12]:
from IPython.core import payload
#Gemini check
import time
def ask_gemini(question, financial_summary=None):
  """Send a finance question to Gemini and return its answer."""
  if not isinstance(question, str) or not question.strip():
    return "Enter a question."
  if not GEMINI_API_KEY:
    return "Error: GEMINI Key not found, maybe you dropped it :P"
  if financial_summary is not None:
    prompt = f"""
User's financial data:
{financial_summary}
User's question:
{question}
"""
  else:
    prompt = question
  url = ("https://generativelanguage.googleapis.com/v1beta/"f"models/{GEMINI_MODEL}:generateContent")
  headers = {"x-goog-api-key": GEMINI_API_KEY, "Content-Type": "application/json"}
  payload = {"system_instruction":{"parts":[{"text": FINANCE_PERSONA}]}, "contents": [{"parts": [{"text": prompt}]}]}
  for attempt in range(3):
    try:
            response = requests.post(
                url,
                headers=headers,
                json=payload,
                timeout=45
            )

            if response.status_code in [503, 500, 429]:
                if attempt < 2:
                    wait_seconds = 2 ** (attempt + 1)
                    print(
                        f"Gemini temporarily unavailable. "
                        f"Retrying in {wait_seconds} seconds..."
                    )
                    time.sleep(wait_seconds)
                    continue

                return (
                    f"Gemini is still unavailable "
                    f"(HTTP {response.status_code}). "
                    "Please try again later."
                )

            response.raise_for_status()
            data = response.json()

            return data["candidates"][0]["content"]["parts"][0]["text"]

    except requests.exceptions.Timeout:
            if attempt < 2:
                time.sleep(2 ** (attempt + 1))
                continue
            return "Gemini timed out after several attempts. Try again."

    except requests.exceptions.HTTPError as error:
            # Show Google's error message without revealing your API key
            try:
                details = response.json().get("error", {}).get(
                    "message", str(error)
                )
            except ValueError:
                details = str(error)

            return f"Gemini API error: {details}"

    except (KeyError, IndexError, TypeError):
            return "Gemini returned an unexpected response."

    except requests.exceptions.RequestException:
            if attempt < 2:
                time.sleep(2 ** (attempt + 1))
                continue
            return "Unable to connect to Gemini. Please try again."

    return "Gemini could not complete the request."

In [21]:
#Gemini Analysis instead of coding anaylsis
example_df = pd.DataFrame([
    {"Type": "Income", "Amount": 3000.0, "Category": "Salary"},
    {"Type": "Expense", "Amount": 800.0, "Category": "Rent"},
    {"Type": "Expense", "Amount": 250.0, "Category": "Groceries"},
    {"Type": "Expense", "Amount": 100.0, "Category": "Transport"}])
summary = calculate_budget_breakdown(example_df, 1500)
answer = ask_gemini("Am I staying within my monthly budget? What's my position.", financial_summary=summary)
print(answer)

Well, look at you being financially responsible! Let’s give credit where credit is due: **Yes, you are staying within your monthly budget.** 

Here is the full breakdown of your financial position:

---

### **1. The Budget Breakdown**
* **Monthly Budget:** $1,500.00  
* **Total Expenses:** $1,150.00  
* **Budget Remaining:** **$350.00**  
* **Budget Used:** **76.67%**  

**The Math:**  
$$\$1,500.00 \text{ (Budget)} - \$1,150.00 \text{ (Expenses)} = \$350.00 \text{ left to spend}$$  

You’ve used about three-quarters of your budget, meaning you have **$350.00** left before you hit the limit you set for yourself. 

Here’s where that $1,150.00 went:
* **Rent:** $800.00 *(the lion's share, as usual!)*
* **Groceries:** $250.00
* **Transport:** $100.00

---

### **2. Your Overall Position (The Big Picture)**
Your budget looks good, but your overall cash flow looks even better:

* **Total Income:** $3,000.00  
* **Total Expenses:** $1,150.00  
* **Remaining Balance:** **$1,850.00**  

**The

In [24]:
#Gradio upload
import gradio as gr
import pandas as pd

def read_gradio_csv(filepath):
  """Read and clean a CSV file upload through Gradio."""
  if not filepath:
    raise ValueError("Please upload a CSV file first.")
  required = ["Date", "Category", "Description", "Type", "Amount"]
  df = None
  for encoding in ["utf-8", "cp1252", "latin1"]:
    try:
      df = pd.read_csv(filepath, encoding=encoding)
      break
    except UnicodeDecodeError:
      continue
    except pd.errors.ParserError:
      raise ValueError("The CSV format could not be read.")
    if not all(column in df.columns for column in required):
      raise ValueError("The CSV is missing required columns: "+", ".join(required))
    df = df[required].copy()
    df["Amount"] = (df["Amount"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.strip())
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Type"] = df["Type"].astype(str).str.strip().str.title()
    df["Category"] = df["Category"].astype(str).str.strip()
    df["Description"] = df["Description"].fillna("").astype(str).str.strip()
    invalid = (
        df["Date"].isna()
        | df["Amount"].isna()
        | (df["Amount"] <= 0)
        | (~df["Type"].isin(["Income", "Expense"]))
        | (df["Category"] == ""))
    if invalid.any():
      raise ValueError(f"{int(invalid.sum())} invalid transaction row(s) found. Please correct the dates, amounts, types and categories in your CSV and upload it again.")
      return df

def analyse_for_gradio(source, filepath, manual_rows, monthly_budget):
    """Analyse uploaded or manually entered transactions."""
    try:
        if monthly_budget is None or monthly_budget < 0:
            raise ValueError("Please enter a valid non-negative budget.")
        if source == "Upload CSV":
            df = read_gradio_csv(filepath)
        else:
            if manual_rows is None:
                raise ValueError("Please enter at least one transaction.")
            df = pd.DataFrame(
                manual_rows,
                columns=["Date", "Category", "Description", "Type", "Amount"])
            df = df.dropna(how="all")
            if df.empty:
                raise ValueError("Please enter at least one transaction.")
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            df["Category"] = df["Category"].fillna("").astype(str).str.strip()
            df["Description"] = (
                df["Description"].fillna("").astype(str).str.strip())
            df["Type"] = df["Type"].fillna("").astype(str).str.strip().str.title()
            df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
            invalid = (
                df["Date"].isna()
                | df["Amount"].isna()
                | (df["Amount"] <= 0)
                | (~df["Type"].isin(["Income", "Expense"]))
                | (df["Category"] == ""))
            if invalid.any():
                raise ValueError(
                    "Some manual entries are invalid. Check the date, "
                    "category, transaction type and positive amount. "
                    "Use Income or Expense for the type.")
        result = calculate_budget_breakdown(df, float(monthly_budget))
        summary = (
            f"Total income: ${result['total_income']:,.2f}\n"
            f"Total expenses: ${result['total_expenses']:,.2f}\n"
            f"Income minus expenses: ${result['balance']:,.2f}\n"
            f"Monthly budget: ${result['monthly_budget']:,.2f}\n"
            f"Budget remaining: ${result['budget_remaining']:,.2f}\n"
            f"Budget used: {result['budget_used_percentage']:.2f}%")
        if result["spending_by_category"]:
            category_text = "\n".join(
                f"{category}: ${amount:,.2f}"
                for category, amount in sorted(
                    result["spending_by_category"].items(),
                    key=lambda item: item[1],
                    reverse=True))
        else:
            category_text = "No expenses recorded."
        return summary, category_text, df, df
    except (ValueError, TypeError, KeyError) as error:
        return f"Could not analyse transactions: {error}", "", None, None

def chat_about_finances(question, df, monthly_budget):
    """Give Gemini the user's actual financial summary."""
    if df is None:
        return "Please analyse your transactions before asking a question."
    try:
        summary = calculate_budget_breakdown(df, float(monthly_budget))
        return ask_gemini(question, financial_summary=summary)
    except (ValueError, TypeError, KeyError) as error:
        return f"Could not prepare your financial summary: {error}"
with gr.Blocks(title="Personal Finance Assistant") as demo:
    gr.Markdown("# Personal Finance Assistant")
    gr.Markdown(
        "Track your money, analyse your budget and ask Gemini "
        "questions about your financial situation.")
    transaction_state = gr.State(None)
    with gr.Row():
        source = gr.Radio(
            choices=["Manual Entry", "Upload CSV"],
            value="Manual Entry",
            label="How would you like to enter your finances?")
        monthly_budget = gr.Number(
            value=1500,
            minimum=0,
            label="Monthly spending budget ($)")
    gr.Markdown(
        "For manual entry, fill in the table below. "
        "For CSV uploads, use the same column names shown in the table.")
    csv_file = gr.File(
        label="Upload your transactions CSV",
        file_types=[".csv"],
        type="filepath")
    manual_table = gr.Dataframe(
        headers=["Date", "Category", "Description", "Type", "Amount"],
        datatype=["str", "str", "str", "str", "number"],
        value=[
            ["2026-10-01", "Salary", "Monthly pay", "Income", 3000],
            ["2026-10-02", "Rent", "Monthly rent", "Expense", 800],
            ["2026-10-03", "Groceries", "Food shopping", "Expense", 250]],
        row_count=(3, "dynamic"),
        column_count=(5, "fixed"),
        interactive=True,
        label="Transaction entries")
    analyse_button = gr.Button("Analyse My Finances", variant="primary")
    gr.Markdown("## Your financial summary")
    summary_output = gr.Textbox(
        label="Budget and balance",
        lines=6,
        interactive=False)
    category_output = gr.Textbox(
        label="Spending by category",
        lines=5,
        interactive=False)
    history_output = gr.Dataframe(
        label="Transactions analysed",
        interactive=False)
    analyse_button.click(
        fn=analyse_for_gradio,
        inputs=[source, csv_file, manual_table, monthly_budget],
        outputs=[
            summary_output,
            category_output,
            history_output,
            transaction_state])
    gr.Markdown("## Ask your AI finance assistant")
    question_input = gr.Textbox(
        label="Your question",
        placeholder="e.g. How can I reduce my spending?",
        lines=2)
    ask_button = gr.Button("Ask Gemini")
    answer_output = gr.Markdown()
    ask_button.click(
        fn=chat_about_finances,
        inputs=[question_input, transaction_state, monthly_budget],
        outputs=answer_output)
demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://94bc86c210fb3a601c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
